# Foldy–Wouthuysen Derivation: The $H_{\mu\nu}$ SME Coefficient

**MSc Research — Oyewo Temidayo Solomon**  
University of Ibadan | Supervisor: Prof. O.E. Oyewande

---

## Overview

This notebook derives the nonrelativistic Hamiltonian from the CPT-even,
Lorentz-violating antisymmetric tensor coefficient $H_{\mu\nu}$:

$$\mathcal{L}_H = -\frac{1}{2}H_{\mu\nu}\bar{\psi}\sigma^{\mu\nu}\psi, \qquad \sigma^{\mu\nu} = \frac{i}{2}[\gamma^\mu, \gamma^\nu]$$

$H_{\mu\nu}$ is real, antisymmetric ($H_{\mu\nu} = -H_{\nu\mu}$), with 6 independent components.  
It is **CPT-even** — the antiparticle Hamiltonian has the **same sign** as the particle one.

**Steps:**
1. Decompose $H_{\mu\nu}$ into electric-like ($H_{0i}$) and magnetic-like ($H_{ij}$) parts
2. Express $\sigma^{\mu\nu}$ in terms of $\alpha^i$ and $\Sigma^i$
3. Classify even/odd operators
4. Apply FW transformation and compute $\mathcal{O}^2$
5. Extract $H^\text{NR}$ and match to Dobrescu–Mocioiu $V_3$, $V_7$


## 1. Setup

In [ ]:
import sys, os
sys.path.insert(0, os.path.dirname(os.path.abspath('__file__')))

import sympy as sp
from sympy import I, Matrix, symbols, Rational, simplify, pprint, zeros, eye, sqrt

try:
    from dirac_algebra import (
        beta, alpha, gamma, gamma5, Sigma,
        sigma_munu, comm, acomm, is_even, is_odd,
        even_part, odd_part, upper, lower, I4, Z4
    )
    from pauli_matrices import sigma, I2, Z2, H_NR_Hmunu, cpt_conjugate
    print("Modules loaded.")
except ImportError:
    print("Defining inline...")
    # Same inline fallback as FW_bmu_term.ipynb
    sigma_x = Matrix([[0,1],[1,0]])
    sigma_y = Matrix([[0,-I],[I,0]])
    sigma_z = Matrix([[1,0],[0,-1]])
    sigma = [sigma_x, sigma_y, sigma_z]
    I2 = eye(2); Z2 = zeros(2,2); I4 = eye(4); Z4 = zeros(4,4)
    beta  = Matrix([[1,0,0,0],[0,1,0,0],[0,0,-1,0],[0,0,0,-1]])
    alpha = [
        Matrix([[0,0,0,1],[0,0,1,0],[0,1,0,0],[1,0,0,0]]),
        Matrix([[0,0,0,-I],[0,0,I,0],[0,-I,0,0],[I,0,0,0]]),
        Matrix([[0,0,1,0],[0,0,0,-1],[1,0,0,0],[0,-1,0,0]]),
    ]
    gamma5 = Matrix([[0,0,1,0],[0,0,0,1],[1,0,0,0],[0,1,0,0]])
    Sigma  = [
        Matrix([[0,1,0,0],[1,0,0,0],[0,0,0,1],[0,0,1,0]]),
        Matrix([[0,-I,0,0],[I,0,0,0],[0,0,0,-I],[0,0,I,0]]),
        Matrix([[1,0,0,0],[0,-1,0,0],[0,0,1,0],[0,0,0,-1]]),
    ]
    def comm(A,B): return A*B-B*A
    def is_even(M): return simplify(beta*M-M*beta)==Z4
    def is_odd(M):  return simplify(beta*M+M*beta)==Z4
    def upper(M):   return M[:2,:2]
    def lower(M):   return M[2:,2:]
    def sigma_munu(mu,nu):
        glist = [beta,
                 Matrix([[0,0,0,1],[0,0,1,0],[0,1,0,0],[1,0,0,0]]),   # g1
                 Matrix([[0,0,0,-I],[0,0,I,0],[0,-I,0,0],[I,0,0,0]]),  # g2
                 Matrix([[0,0,1,0],[0,0,0,-1],[1,0,0,0],[0,-1,0,0]])]  # g3
        return Rational(1,2)*I*(glist[mu]*glist[nu]-glist[nu]*glist[mu])


## 2. Decomposing $\sigma^{\mu\nu}$

The 6 independent components of $H_{\mu\nu}$ split into:

- **Electric-like** (boost generators): $H_{0i}$, contributing via $\sigma^{0i} = i\alpha^i$  
- **Magnetic-like** (rotation generators): $H_{ij}$, contributing via $\sigma^{ij} = \epsilon^{ijk}\Sigma^k$

We verify these algebraic identities explicitly.


In [ ]:
print("=== Verifying sigma^{mu nu} decomposition ===")
print()

# sigma^{0i} = i * alpha^i
for i in range(1, 4):
    s0i = sigma_munu(0, i)
    expected = I * alpha[i-1]
    match = simplify(s0i - expected) == Z4
    print(f"sigma^{{0{i}}} = i*alpha[{i-1}]? {match}")
    if i == 1:
        print("  sigma^{01} ="); pprint(s0i)

print()
# sigma^{ij} = epsilon^{ijk} Sigma^k
eps = {(1,2):( 3, 1), (2,3):( 1, 1), (1,3):( 2,-1),
       (2,1):( 3,-1), (3,2):( 1,-1), (3,1):( 2, 1)}
for (i,j),(k,e) in eps.items():
    sij = sigma_munu(i, j)
    expected = e * Sigma[k-1]
    match = simplify(sij - expected) == Z4
    print(f"sigma^{{{i}{j}}} = {e:+d}*Sigma[{k-1}]? {match}")


## 3. The $H_{\mu\nu}$ Hamiltonian Contribution

$$H_H = -\frac{1}{2}H_{\mu\nu}\gamma^0\sigma^{\mu\nu}$$

Substituting the decomposed forms:

$$H_H = -iH_{0i}\beta\alpha^i - \frac{1}{2}H_{ij}\epsilon^{ijk}\beta\Sigma^k$$

Defining $\mathcal{H}_B^k = \frac{1}{2}\epsilon^{ijk}H_{ij}$:

$$H_H = -i\mathbf{H}_E\cdot\boldsymbol{\alpha}\beta \quad (\text{odd}) \quad - \mathbf{H}_B\cdot\boldsymbol{\Sigma}\beta \quad (\text{even})$$


In [ ]:
def _msum(terms):
    """Matrix-safe sum."""
    from sympy import zeros
    result = None
    for t in terms:
        result = t if result is None else result + t
    return result if result is not None else zeros(4,4)

# Define H_munu components symbolically
H_01, H_02, H_03 = symbols('H_01 H_02 H_03', real=True)  # electric-like
H_12, H_13, H_23 = symbols('H_12 H_13 H_23', real=True)  # magnetic-like

H_E = [H_01, H_02, H_03]

# Magnetic-like: H_B^k = (1/2) epsilon^{ijk} H_{ij}
# H_B^1 = H_23, H_B^2 = H_31 = -H_13, H_B^3 = H_12
H_B = [H_23, -H_13, H_12]

print("H_E (electric-like) =", H_E)
print("H_B (magnetic-like) =", H_B)

# Odd part: -i H_E . alpha . beta
H_H_odd = _msum([-I * H_E[i] * alpha[i] * beta for i in range(3)])
print("\nOdd part H_H (odd)  [-i H_E.alpha.beta]:")
pprint(H_H_odd)
print("Is odd?", is_odd(H_H_odd))

# Even part: -H_B . Sigma . beta
H_H_even = _msum([-H_B[i] * Sigma[i] * beta for i in range(3)])
print("\nEven part H_H (even) [-H_B.Sigma.beta]:")
pprint(H_H_even)
print("Is even?", is_even(H_H_even))


## 4. FW Transformation

The total odd operator (including kinetic term) is:

$$\mathcal{O} = \boldsymbol{\alpha}\cdot\mathbf{p} + \mathcal{O}_H, \qquad \mathcal{O}_H = -i\mathbf{H}_E\cdot\boldsymbol{\alpha}\beta$$

We can write $\mathcal{O} = \boldsymbol{\alpha}\cdot(\mathbf{p} - i\mathbf{H}_E\beta)$.

The FW generator $S = -i\beta\mathcal{O}/(2m)$ cancels $\mathcal{O}$ at leading order.


In [ ]:
def _msum(terms):
    """Matrix-safe sum."""
    from sympy import zeros
    result = None
    for t in terms:
        result = t if result is None else result + t
    return result if result is not None else zeros(4,4)

p1, p2, p3, m_sym = symbols('p_1 p_2 p_3 m', real=True)
p_vec = [p1, p2, p3]

# Total odd operator
alpha_dot_p = _msum([p_vec[i] * alpha[i] for i in range(3)])
O_total_H = alpha_dot_p + H_H_odd
print("Total odd operator O = alpha.p + O_H:")
print("(showing upper-left 2x2 block for readability)")
pprint(O_total_H[:2,:2])

# FW generator
S_H = -I * beta * O_total_H / (2 * m_sym)

# Verify cancellation at leading order
check = simplify(I * comm(S_H, beta * m_sym) + O_total_H)
print("\nVerification i[S,beta*m] + O = 0?", check == Z4)


## 5. Computing $\mathcal{O}^2$

With $\mathcal{O} = \boldsymbol{\alpha}\cdot(\mathbf{p} - i\mathbf{H}_E\beta)$, expand to first order in $H_{\mu\nu}$:

$$\mathcal{O}^2 = \mathbf{p}^2 - 2i(\mathbf{p}\cdot\mathbf{H}_E)\beta - 2\boldsymbol{\Sigma}\cdot(\mathbf{p}\times\mathbf{H}_E)\beta + O(H^2)$$

The second term is a pseudo-scalar (even, real); the third generates a spin-orbit-like structure.


In [ ]:
def _msum(terms):
    """Matrix-safe sum."""
    from sympy import zeros
    result = None
    for t in terms:
        result = t if result is None else result + t
    return result if result is not None else zeros(4,4)

# O^2 — compute and display upper block
O_sq_H = simplify(O_total_H * O_total_H)

# Expected structure (to 1st order in H_E):
# p^2 * I4  -  2i (p.H_E) beta  -  2 Sigma.(p x H_E) beta
p_sq = p1**2 + p2**2 + p3**2

# p x H_E
pxH_E = [
    p2*H_E[2] - p3*H_E[1],
    p3*H_E[0] - p1*H_E[2],
    p1*H_E[1] - p2*H_E[0],
]
Sigma_dot_pxH = _msum([pxH_E[i]*Sigma[i] for i in range(3)])
pdotH = p1*H_E[0] + p2*H_E[1] + p3*H_E[2]

# Leading-order expected (to 1st order in H_E, ignoring H_E^2)
O_sq_expected = (p_sq * I4
                 - 2*I*pdotH*beta
                 - 2*Sigma_dot_pxH*beta)

# Difference (should vanish to first order in H)
diff = simplify(O_sq_H - O_sq_expected)
# Check only terms linear in H_E (drop H^2 terms)
diff_linear = diff.applyfunc(
    lambda x: sum(
        x.coeff(h)*h for h in [H_01,H_02,H_03,H_12,H_13,H_23]
    )
)
print("O^2 - expected (linear in H_E only):")
pprint(diff_linear)
print("Linear-order match?", simplify(diff_linear) == Z4)


## 6. Nonrelativistic Hamiltonian — Upper Components

For the upper (particle) sector:

$$H^\text{NR}_{H_{\mu\nu}} = \frac{\mathbf{p}^2}{2m} - \mathbf{H}_B\cdot\boldsymbol{\sigma} - \frac{1}{m}\boldsymbol{\sigma}\cdot(\mathbf{p}\times\mathbf{H}_E)$$

The two spin-dependent terms:
- $-\mathbf{H}_B\cdot\boldsymbol{\sigma}$: dipole-like splitting, maps to **V3**
- $-\frac{1}{m}\boldsymbol{\sigma}\cdot(\mathbf{p}\times\mathbf{H}_E)$: velocity-dependent spin-orbit, maps to **V7**


In [ ]:
# Upper block of O^2 / 2m
O_sq_upper = simplify(O_sq_H[:2, :2])
H_from_O2_upper = simplify(O_sq_upper / (2*m_sym))

print("Upper block of O^2/(2m):")
pprint(H_from_O2_upper)

# Even part contributes directly: upper block of -H_B.Sigma.beta
# (beta -> +1 for upper components)
H_H_even_upper = simplify(H_H_even[:2, :2])
print("\nEven contribution (upper block of -H_B.Sigma.beta):")
pprint(H_H_even_upper)

# Total NR Hamiltonian (spin-dependent, dropping p^2/2m)
p_sq_term = (p1**2+p2**2+p3**2)/(2*m_sym) * I2
H_NR_spin_H = simplify(H_from_O2_upper + H_H_even_upper - p_sq_term)
print("\nSpin-dependent H^NR_{H_munu} (leading order):")
pprint(H_NR_spin_H)


In [ ]:
# Use pauli_matrices.py convenience function
try:
    H_check, desc = H_NR_Hmunu(H_B, H_E=[H_01,H_02,H_03], p=[p1,p2,p3])
    print("From pauli_matrices.H_NR_Hmunu:")
    print(desc)
    pprint(H_check)
except Exception as ex:
    print("pauli_matrices not available:", ex)


## 7. CPT Properties

$H_{\mu\nu}$ is **CPT-even**. Under charge conjugation:

$$H^\text{NR}_{H,\,\bar{f}} = +H^\text{NR}_{H,\,f}$$

The antiparticle Hamiltonian has the **same sign**. This means:
- Both matter and antimatter precess in the **same direction** in a $H_{\mu\nu}$ background.
- No asymmetry $A_\alpha$ is expected from $H_{\mu\nu}$ alone.
- Any observed $A_\alpha \neq 0$ in an $H_{\mu\nu}$-sensitive experiment signals additional CPT-odd contributions.


In [ ]:
# Lower block of H_H_even (antiparticle, beta -> -1)
H_H_even_lower = simplify(H_H_even[2:, 2:])
print("Even part [lower block, beta=-1]:")
pprint(H_H_even_lower)
print()
print("Lower block of O^2/(2m) [spin terms only, beta=-1]:")
O_sq_lower = simplify(O_sq_H[2:, 2:])
pprint(simplify(O_sq_lower / (2*m_sym)))
print()
print("=> For antimatter: H^NR_{H,fbar} = -H_B.sigma  [SAME as matter]")
print("   CPT-even: no sign flip. Expected A_alpha = 0.")


## 8. Matching to Dobrescu–Mocioiu Potentials

| $H_{\mu\nu}$ component | NR structure | DM potential | Physical effect |
|---|---|---|---|
| $H_{ij}$ (magnetic-like) | $-\mathbf{H}_B\cdot\boldsymbol{\sigma}$ | $V_3$ | Dipole-dipole, CPT-even |
| $H_{0i}$ (electric-like) | $-\frac{1}{m}\boldsymbol{\sigma}\cdot(\mathbf{p}\times\mathbf{H}_E)$ | $V_7$ | Spin-velocity, CPT-even |

For a two-body problem where the source particle $1$ also has spin $\boldsymbol{\sigma}_1$,
the field $\mathbf{H}_B$ is generated by $\boldsymbol{\sigma}_1$, giving the full dipole-dipole form of $V_3$.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Visualise: spin splitting from H_B field vs direction
theta = np.linspace(0, 2*np.pi, 300)
H_B_mag = 1.0

# Energy eigenvalues: +/- |H_B|
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

ax = axes[0]
ax.fill_between(theta*180/np.pi, -H_B_mag, H_B_mag,
                alpha=0.15, color='steelblue', label='Allowed energies')
ax.plot(theta*180/np.pi, np.full_like(theta,  H_B_mag), 'steelblue', lw=2,
        label=r'$E_+ = +|\mathbf{H}_B|$')
ax.plot(theta*180/np.pi, np.full_like(theta, -H_B_mag), 'steelblue', lw=2,
        ls='--', label=r'$E_- = -|\mathbf{H}_B|$')
ax.plot(theta*180/np.pi, np.full_like(theta,  H_B_mag), 'tomato', lw=2,
        ls=':', alpha=0.8, label=r'Antimatter: $E_+ = +|\mathbf{H}_B|$ (same!)')
ax.plot(theta*180/np.pi, np.full_like(theta, -H_B_mag), 'tomato', lw=2,
        ls='-.', alpha=0.8, label=r'Antimatter: $E_- = -|\mathbf{H}_B|$ (same!)')
ax.set_xlabel(r'Orientation angle $	heta$ (deg)', fontsize=11)
ax.set_ylabel(r'Energy / $|\mathbf{H}_B|$', fontsize=11)
ax.set_title(r'$H_{\mu
u}$ (CPT-even): same splitting for matter and antimatter', fontsize=11)
ax.legend(fontsize=8)
ax.grid(alpha=0.3)

# Right: compare CPT-odd (b_mu) vs CPT-even (H_munu) asymmetry prediction
ax2 = axes[1]
categories = [r'$b_\mu$ (CPT-odd)', r'$H_{\mu
u}$ (CPT-even)', r'$d_{\mu
u}$ (CPT-even)']
A_theory = [1.0, 0.0, 0.0]
A_observed_min = [0.954, None, None]  # from SPINDEP report

x = np.arange(len(categories))
bars = ax2.bar(x, A_theory, color=['tomato','steelblue','goldenrod'],
               alpha=0.7, width=0.5, label='Theory prediction')
ax2.axhline(0, color='k', lw=0.8, ls='--')
ax2.set_xticks(x)
ax2.set_xticklabels(categories, fontsize=10)
ax2.set_ylabel(r'Predicted $|A_lpha|$', fontsize=11)
ax2.set_title('CPT prediction: asymmetry by coefficient type', fontsize=11)
ax2.set_ylim(-0.1, 1.3)
ax2.annotate(r'SPINDEP: $|A|\geq 0.954$', xy=(0, 1.0), xytext=(0.3, 1.15),
             fontsize=9, color='tomato',
             arrowprops=dict(arrowstyle='->', color='tomato'))
ax2.grid(alpha=0.3, axis='y')

plt.tight_layout()
plt.savefig('FW_Hmunu_CPT_comparison.pdf', dpi=150, bbox_inches='tight')
plt.show()
print("Figure saved: FW_Hmunu_CPT_comparison.pdf")


## Summary

| Quantity | Result |
|---|---|
| SME term | $\mathcal{L}_H = -\frac{1}{2}H_{\mu\nu}\bar\psi\sigma^{\mu\nu}\psi$ |
| CPT property | **Even** — same sign for matter and antimatter |
| Odd operator | $\mathcal{O}_H = -i\mathbf{H}_E\cdot\boldsymbol{\alpha}\beta$ |
| Even operator | $\mathcal{E}_H = -\mathbf{H}_B\cdot\boldsymbol{\Sigma}\beta$ |
| $H^\text{NR}$ (matter) | $-\mathbf{H}_B\cdot\boldsymbol{\sigma} - \frac{1}{m}\boldsymbol{\sigma}\cdot(\mathbf{p}\times\mathbf{H}_E)$ |
| $H^\text{NR}$ (antimatter) | **same** (CPT-even, no sign flip) |
| DM potentials | $V_3$ (dipole-dipole) + $V_7$ (spin-velocity) |
| Expected $|A_\alpha|$ | 0 (no CPT violation from $H_{\mu\nu}$ alone) |
| Implication for thesis | Any $|A_\alpha| \neq 0$ in $V_3/V_7$ sector requires additional CPT-odd source |
